# Toy problem: RV-SDDP versus Cyclic SDDP

This notebook reproduces the numerical experiments on the toy problem presented in the accompanying paper. The problem is a simple infinite-horizon discounted reservoir management problem used to illustrate the convergence behavior of the proposed **RV-SDDP** algorithm and to compare it with **Cyclic SDDP**, i.e., the variant without shifts.

In [ ]:
import Pkg
Pkg.activate(".")

In [ ]:
using Revise
using RVSDDP
using Random
using Plots
using Gurobi
using Statistics
# using HiGHS
# optimizer = () -> HiGHS.Optimizer()
using Gurobi
const GRB_ENV = Gurobi.Env()
optimizer=() -> Gurobi.Optimizer(GRB_ENV)
using LaTeXStrings

## Run RV-SDDP and Cyclic SDDP

In [ ]:
discount_factor=0.99
period = 1
graph=RVSDDP.InfiniteLinearGraph(period);

In [ ]:
function subproblem_builder(subproblem::Model, node::Int, discount_factor::Float64)
    # State variables
    @variable(subproblem, 0 <= volume <= 200, RVSDDP.State, initial_value = 50)
    # Control variables
    @variables(subproblem, begin
        thermal_generation >= 0
        hydro_generation >= 0
        hydro_spill >= 0
        deficit >= 0
    end)
    # Random variables
    @variable(subproblem, inflow)
    Ω = [20.0, 90.0]
    P = [1 / length(Ω) for _ in Ω]
    RVSDDP.parameterize(subproblem, Ω, P) do ω
        return JuMP.fix(inflow, ω)
    end

    # Transition function and constraints
    @constraints(
        subproblem,
        begin
            volume.out == volume.in - hydro_generation - hydro_spill + inflow
            hydro_generation <= 100
            thermal_generation <= 30
            deficit + hydro_generation + thermal_generation == 60
        end
    )
    # Stage-objective
    @stageobjective(subproblem, 50*hydro_spill + 50 * deficit+ 2*thermal_generation)
    return subproblem
end

In [ ]:
model_cyclic_sddp = RVSDDP.PolicyGraph(
    subproblem_builder,
    graph;
    sense = :Min,
    lower_bound = 0.0,
    optimizer = optimizer,
    discount_factor=discount_factor,
)

Random.seed!(1)
parallel= 1
Cuts0=RVSDDP.train(model_cyclic_sddp; refine_mode = 0, parallel=parallel, sampling_scheme=RVSDDP.InSampleMonteCarlo(max_depth=10000, rollout_limit = i -> period*i, parallel=parallel), iteration_limit = 100, infinite = true, shift_function=RVSDDP.no_shift); 

In [ ]:
model_RVSDDP = RVSDDP.PolicyGraph(
    subproblem_builder,
    graph;
    sense = :Min,
    lower_bound = 0.0,
    optimizer = optimizer,
    discount_factor=discount_factor,
)

Random.seed!(1)
parallel= 1
Cuts0=RVSDDP.train(model_RVSDDP; refine_mode = 0, parallel=parallel, sampling_scheme=RVSDDP.InSampleMonteCarlo(max_depth=10000, rollout_limit = i -> period*i, parallel=parallel), iteration_limit = 100, infinite = true, shift_function=RVSDDP.shift_update_random_forward); 

## Comparison of value function approximations obtained by RV-SDDP and cyclic SDDP after 100 iterations.

In [ ]:
fontsize = 20
ind = 0:0.5:200
V_cyclic_sddp = [RVSDDP.compute_V(model_cyclic_sddp[1].value_function, Dict(:volume=>1.0*i)) for i in ind]
TV_cyclic_sddp = [RVSDDP.compute_TV(model_cyclic_sddp[1], Dict(:volume=>1.0*i)) for i in ind]
V_RVSDDP = [RVSDDP.compute_V(model_RVSDDP[1].value_function, Dict(:volume=>1.0*i)) for i in ind]
TV_RVSDDP = [RVSDDP.compute_TV(model_RVSDDP[1], Dict(:volume=>1.0*i)) for i in ind]

p1 = plot(
    ind,
    V_cyclic_sddp,
    label = false,
    xlabel = L"x",
    guidefontsize = fontsize,
    tickfontsize = fontsize,
    linewidth = 3,
    margin = 10Plots.mm,
    title = "Value function",
    titlefontsize = fontsize
)
plot!(p1, ind, V_RVSDDP, label = false, linewidth = 3)

p2 = plot(
    ind,
    TV_cyclic_sddp - V_cyclic_sddp,
    label = false,
    xlabel = L"x",
    guidefontsize = fontsize,
    tickfontsize = fontsize,
    linewidth = 3,
    margin = 10Plots.mm,
    title = "Bellman residuals",
    titlefontsize = fontsize
)
plot!(p2, ind, TV_RVSDDP - V_RVSDDP, label = false, linewidth = 3)

p3=plot(label = false, xlabel=L"n",guidefontsize=fontsize, tickfontsize=fontsize, legendfontsize=fontsize, margin=10Plots.mm, 
    title = "Shift",
    titlefontsize = fontsize)

plot!(p3, [cut.shift[1][1] for cut in model_RVSDDP[1].value_function.cut_V], xticks = [0, 1500,3000,4500],label=false, linewidth=3, color = 2)

pleg = plot(
    [NaN], [NaN],
    label = "Cyclic SDDP",
    linewidth = 3,
    color = 1,
    framestyle = :none,
    legend = :bottom,
    legend_columns = 3,
    legendfontsize = fontsize,
    ticks = false,
    grid = false,
)
plot!(
    pleg,
    [NaN], [NaN],
    label = " ",
    linewidth = 3,
    color = :white,
)
plot!(
    pleg,
    [NaN], [NaN],
    label = "RV-SDDP",
    linewidth = 3,
    color = 2,
)

plot(
    p1,
    p2,
    p3,
    pleg,
    layout = @layout([a b c; d{0.2h}]),
    size = (1800, 700),
)

## Value function approximations produced by cyclic SDDP at different iterations.

In [ ]:
iter_max = 100
model_cyclic_sddp_list = [RVSDDP.PolicyGraph(
        subproblem_builder,
        graph;
        sense = :Min,
        lower_bound = 0.0,
        optimizer = optimizer,
        discount_factor=discount_factor,
    ) for iter in 10:10:iter_max]

for (i, iter) in enumerate(10:10:iter_max)
    RVSDDP.add_cuts_to_model(model_cyclic_sddp_list[i], model_cyclic_sddp, iter);
end

model_RVSDDP_list = [RVSDDP.PolicyGraph(
        subproblem_builder,
        graph;
        sense = :Min,
        lower_bound = 0.0,
        optimizer = optimizer,
        discount_factor=discount_factor,
    ) for iter in 10:10:iter_max]

for (i, iter) in enumerate(10:10:iter_max)
    RVSDDP.add_cuts_to_model(model_RVSDDP_list[i], model_RVSDDP, iter);
end

ind = 0:1:200

fontsize = 20

p1=plot(xlabel=L"x", ylabel = L"V^\Delta(x)", guidefontsize=fontsize, tickfontsize=fontsize, titlefontsize = fontsize,legendfontsize=fontsize, size=(1000, 700), margin=10Plots.mm,title = "Cyclic SDDP")
color=10
for i in 1:10
    iter = i*10
    V_cyclic_sddp = [RVSDDP.compute_V(model_cyclic_sddp_list[i][1].value_function, Dict(:volume=>1.0*j)) for j in ind]
    plot!(p1, ind, V_cyclic_sddp, label = false, linewidth = 3, color = color)
    color -= 1
end

p2=plot(xlabel=L"x", guidefontsize=fontsize, tickfontsize=fontsize, titlefontsize = fontsize,legendfontsize=fontsize, size=(1000, 700), margin=10Plots.mm, ylims = (-100, 2700),title = "Cyclic SDDP normalized")
color=10
for i in 1:10
    iter = i*10
    Vref = RVSDDP.compute_V(model_cyclic_sddp_list[i][1].value_function, Dict(:volume=>30.0))
    V_cyclic_sddp = [RVSDDP.compute_V(model_cyclic_sddp_list[i][1].value_function, Dict(:volume=>1.0*j))-Vref for j in ind]
    plot!(p2, ind, V_cyclic_sddp, label = false, linewidth = 3, color = color)
    color -= 1
end
p3=plot(xlabel=L"x", guidefontsize=fontsize, tickfontsize=fontsize, titlefontsize = fontsize,legendfontsize=fontsize, size=(1000, 700), margin=10Plots.mm, ylims = (-100, 2700),title = "RV-SDDP")
color=10
for i in 1:10
    iter = i*10
    V_RVSDDP = [RVSDDP.compute_V(model_RVSDDP_list[i][1].value_function, Dict(:volume=>1.0*j)) for j in ind]
    plot!(p3, ind, V_RVSDDP, label = false, linewidth = 3, color = color)
    color -= 1
end

pleg = plot(
    [NaN], [NaN],
    label = " 100 ($(length(model_RVSDDP_list[10][1].value_function.cut_V)))",
    linewidth = 3,
    color = 1,
    framestyle = :none,
    legend = :left,
    legend_columns = 1,
    legendfontsize = fontsize,
    ticks = false,
    grid = false,
    showaxis = false,
    foreground_color_legend = nothing,
    background_color_legend = nothing,
    extra_kwargs = Dict(:subplot => Dict(:legend_hfactor => 1.3)),
    legend_title = "Iteration (# cuts)",
    legend_title_font_pointsize = fontsize,
)

color = 2
for k in 2:10
    i = 10 - k + 1
    iter = i * 10
    ncuts = length(model_RVSDDP_list[i][1].value_function.cut_V)

    plot!(
        pleg,
        [NaN], [NaN],
        label = " $iter ($ncuts)",
        linewidth = 3,
        color = color,
    )

    color += 1
end

p_final = plot(
    p1,
    p2,
    p3,
    pleg,
    layout = @layout([a{0.25w} b{0.25w} c{0.25w} d{0.25w}]),
    size = (1700, 500),
)
p_final

## Policy quality and number of active cuts after 31 iterations

The policy quality is evaluated by Monte Carlo simulation over \(N\) scenarios. For \(N=1000\), one evaluation takes approximately 2 minutes. The estimated policy costs are 7382 for Cyclic SDDP and 6942 for RV-SDDP.

In [ ]:
N = 1000
TimeHorizon = Int(ceil(log(0.001)/(log(discount_factor))))

In [ ]:
model_cyclic_sddp = RVSDDP.PolicyGraph(
    subproblem_builder,
    graph;
    sense = :Min,
    lower_bound = 0.0,
    optimizer = optimizer,
    discount_factor=discount_factor,
)

Random.seed!(1)
parallel= 1
Cuts0=RVSDDP.train(model_cyclic_sddp; refine_mode = 0, parallel=parallel, sampling_scheme=RVSDDP.InSampleMonteCarlo(max_depth=10000, rollout_limit = i -> period*i, parallel=parallel), iteration_limit = 31, infinite = true, shift_function=RVSDDP.no_shift); 

In [ ]:
Random.seed!(12345)

simulations_cyclic_sddp= RVSDDP.simulate(
        model_cyclic_sddp,
        N;
        sampling_scheme = RVSDDP.InSampleMonteCarlo(max_depth=TimeHorizon),
)
println("Average cost for cyclic SDDP: ", mean([sum((discount_factor^(t-1))*simulations_cyclic_sddp[k][t][:stage_objective] for t in 1:TimeHorizon) for k in 1:N]))

In [ ]:
model_rvsddp = RVSDDP.PolicyGraph(
    subproblem_builder,
    graph;
    sense = :Min,
    lower_bound = 0.0,
    optimizer = optimizer,
    discount_factor=discount_factor,
)

Random.seed!(1)
parallel= 1
Cuts0=RVSDDP.train(model_rvsddp; refine_mode = 0, parallel=parallel, sampling_scheme=RVSDDP.InSampleMonteCarlo(max_depth=10000, rollout_limit = i -> period*i, parallel=parallel), iteration_limit = 31, infinite = true, shift_function=RVSDDP.shift_update_random_forward); 

In [ ]:
Random.seed!(12345)

simulations_rvsddp= RVSDDP.simulate(
        model_rvsddp,
        N;
        sampling_scheme = RVSDDP.InSampleMonteCarlo(max_depth=TimeHorizon),
)
println("Average cost for RVSDDP: ", mean([sum((discount_factor^(t-1))*simulations_rvsddp[k][t][:stage_objective] for t in 1:TimeHorizon) for k in 1:N]))